# Análisis Exploratorio de Datos - Campañas de Marketing Bancario

El objetivo de este proyecto es limpiar y analizar los datos de campañas de marketing de una entidad bancaria para identificar patrones y factores que influyen en la contratación de productos.

1. IMPORTACIÓN DE LIBRERÍAS

Primero, instalar pandas (y openpyxl), matplotlib y seaborn en la terminal con pip install, luego importar las librerías necesarias:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

2. CARGA DE DATOS

In [ ]:
df = pd.read_csv("../data/RAW/bank-additional.csv")
excel = pd.ExcelFile("../data/RAW/customer-details.xlsx")

df1 = pd.read_excel(excel, sheet_name=0)
df2 = pd.read_excel(excel, sheet_name=1)
df3 = pd.read_excel(excel, sheet_name=2)

df_clientes = pd.concat([df1, df2, df3])

3. EXPLORACIÓN INICIAL

En este apartado, el objetivo es tener una imágen clara sobre las BBDD, sus columnas, valores y formatos, para poder realizar la limpieza correspondiente. Para ello, usamos funcionemos como head, info, describe. En este punto, también podemos detectar la cantidad de nulos por columna con los que vamos a trabajar, y así determinar el tratamiento que se le dará a cada uno.

In [ ]:
df.columns

In [ ]:
df_clientes.columns

In [ ]:
df.shape

In [ ]:
df_clientes.shape

In [ ]:
df.head()

In [ ]:
df_clientes.head()

In [ ]:
df.info()

In [ ]:
df_clientes.info()

In [ ]:
df.describe()

In [ ]:
df_clientes.describe()

In [ ]:
df[df.duplicated()]

In [ ]:
df_clientes[df_clientes.duplicated()]

In [ ]:
df.isnull().sum()

In [ ]:
df_clientes.isnull().sum()

4. LIMPIEZA INICIAL

Cuando ya tenemos claro los valores con los que trabajamos, nos centramos en que cada uno tenga el formato correspondiente y eliminamos las columnas que no aporten al análisis.

In [ ]:
# Eliminar la columna "Unnamed: 0", índice innecesario
df = df.drop(columns=["Unnamed: 0"])
df_clientes = df_clientes.drop(columns=["Unnamed: 0"])

In [ ]:
# Cambiar el nombre de las columnas para un mejor manejo de las tablas
df.rename(columns={"id_": "ID"}, inplace=True)
df.rename(columns={"y": "Conversion"}, inplace=True)

In [ ]:
# Convertir la variable objetivo "y" a numérica, donde "yes" se convierte en 1 y "no" en 0 para facilitar el análisis.
df["Conversion"] = df["Conversion"].map({"yes": 1, "no": 0})

In [ ]:
# Convertir los valores de las columnas numéricas a tipo numérico, ya que al leer el CSV se interpretaron como objetos (strings), en el excel los formatos están correctos.
cols = ["cons.price.idx", "cons.conf.idx", "euribor3m"]

for col in cols:
    df[col] = df[col].str.replace(",", ".").astype(float)

# Tuve que reemplazar las comas por puntos para convertir los valores a tipo numérico, ya que en el CSV los números decimales estaban representados con comas, por lo que me daba error al intentar convertirlos directamente a float.

In [ ]:
# Convertir la columna "date" a formato datetime, primero traduciendo los meses del español al inglés para que pandas pueda interpretarlos correctamente.
meses = {
    'enero': 'Jan', 'febrero': 'Feb', 'marzo': 'Mar', 'abril': 'Apr',
    'mayo': 'May', 'junio': 'Jun', 'julio': 'Jul', 'agosto': 'Aug',
    'septiembre': 'Sep', 'octubre': 'Oct', 'noviembre': 'Nov', 'diciembre': 'Dec'
}

df['date_en'] = df['date']
for es, en in meses.items():
    df['date_en'] = df['date_en'].str.replace(es, en, regex=True)

df['date'] = pd.to_datetime(df['date_en'], dayfirst=True)

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.day_name()

df.drop(columns=['date_en'], inplace=True)

# En este punto me dio error al intentar convertir la columna "date" a formato datetime, ya que los meses estaban en español y pandas no los reconocía. Por eso tuve que crear una nueva columna "date_en" donde reemplacé los nombres de los meses en español por sus equivalentes en inglés, para luego poder convertir esa columna a datetime sin problemas. Después extraje el año, mes, día y día de la semana en nuevas columnas para facilitar el análisis temporal. Finalmente eliminé la columna "date_en" ya que solo era un paso intermedio para la conversión.


In [ ]:
# En la tabla de df_clientes, la columna "Dt_Customer" ya está en formato datetime, así que solo necesito extraer el día, mes y año en nuevas columnas para facilitar el análisis temporal.

df_clientes["day"] = df_clientes["Dt_Customer"].dt.day
df_clientes["month"] = df_clientes["Dt_Customer"].dt.month
df_clientes["year"] = df_clientes["Dt_Customer"].dt.year

In [ ]:
# Verificamos los cambios realizados
df.info()

In [ ]:
df.head(2)

In [ ]:
df_clientes.info()

In [ ]:
df_clientes.head(2)

5. MANEJO DE VALORES NULOS Y OUTLIERS

Para terminar con la limpieza de las tablas, es necesario tomar decisiones sobre el manejo de los valores nulos y outliers, ya que el correcto manejo de estos es escenial para el posterior análisis de los mismos.

In [ ]:
# Para menajar los nulos en "age", decidí relacionarlo directamente con "marital", para ello, primero rellené los valores faltantes de "marital" con la moda.

df['marital'].fillna(df['marital'].mode()[0], inplace=True)

In [ ]:
# Para rellenar los valores nulos de "age", utilicé la función transform de pandas para aplicar una función lambda que rellena los valores faltantes con la mediana de "age" dentro de cada grupo de "marital". De esta manera, los valores nulos en "age" se rellenan de manera más precisa, teniendo en cuenta la relación entre el estado civil y la edad.
df['age'] = df.groupby('marital')['age'].transform(lambda x: x.fillna(x.median()))

In [ ]:
# Para menajar los nulos de "job" y "education" decidí reemplazarlos por la palabra "unknown".

df['job'].fillna('unknown', inplace=True)
df['education'].fillna('unknown', inplace=True)

In [ ]:
# Para los nulos de "default", "housing" y "loan", decidí rellenarlos con -1, ya que son variables binarias y no quiero introducir una categoría adicional como "unknown", además de que -1 me permitirá identificarlos fácilmente en el análisis posterior.

df['default'].fillna(-1, inplace=True)
df['housing'].fillna(-1, inplace=True)
df['loan'].fillna(-1, inplace=True)

In [ ]:
# Para manejar los nulos de "cons.price.idx", decidí rellenarlos con la media de la columna, ya que no son muchos nulos y la media es una buena opción para este tipo de variable numérica continua.

df['cons.price.idx'].fillna(df['cons.price.idx'].mean(), inplace=True)

In [ ]:
# Para manejar los nulos de la columna "date", al ser un dato importante para el análisis temporal, y al ser tan poco nulos (248 nulos, el 0.5% del total de filas), decidí eliminar las filas que contienen nulos en esta columna, ya que no afectará significativamente el análisis y me permitirá trabajar con datos completos en las fechas.

df = df.dropna(subset=['date'])

In [ ]:
# Para manejar los nulos en la columna "euribor3m", decidí rellenarlos con el último valor conocido, ordenando primero de forma descendente la columna "cons.price.idx".

df = df.sort_values('cons.price.idx', ascending=False)
df['euribor3m'] = df['euribor3m'].fillna(method='ffill')

In [ ]:
#Comprobamos los nulos después de la limpieza
df.isnull().sum()

6. UNIR LAS TABLAS

Una vez finalizada la limpieza de las tablas, es necesario unirlas por el "ID", para poder extraer mayor información.

In [ ]:
# Realizamos un left join debido a que queremos conservar todas las filas de df (tabla principal) y agregar la información de df_clientes (tabla secundaria) donde coincidan los IDs, pero sin eliminar filas de df que no tengan correspondencia.

df_limpio = df.merge(df_clientes, on='ID', how='left')

In [ ]:
# Verificamos que el merge se haya realizado correctamente y que no haya columnas duplicadas ni conflictos de nombres.
df_limpio.columns

7. DESCARGA ARCHIVOS LIMPIOS

Finalmente, antes de pasar al EDA, descargamos los archivos limpios y los guardamos en una carpeta para tenerlos disponible para futuros análisis. 

In [ ]:
df.to_csv("../data/Procesado/bank-additional-clean.csv", index=False)
df_clientes.to_csv("../data/Procesado/customer-details-clean.csv", index=False)
df_limpio.to_csv("../data/Procesado/tablas_limpias.csv", index=False)

8. ANÁLISIS EXPLORATORIO DE DATOS (EDA)

In [ ]:
# Realizamos un conteo de la variable objetivo "Conversion" para saber cuántos clientes se suscribieron a un producto o servicio.
sns.countplot(data=df_limpio, x='Conversion')
plt.title('Conversión (suscripción al depósito)')
plt.show()

In [ ]:
tasa_conversion = df_limpio['Conversion'].mean() * 100
print(tasa_conversion)

En este gráfico podemos ver que la tasa de converción es baja, debido a que la columna 1 (Si) es mucho más chica que la columna 0 (No). Además, si calculamos el porcentaje de conversión, podemos ver que el mismo es del 11,25%.

In [ ]:
#En este gráfico comparamos la distribución de edades entre los clientes que se suscribieron (Conversion=1) y los que no se suscribieron (Conversion=0).

sns.histplot(data=df_limpio, x='age', hue='Conversion', bins=30, kde=True)
plt.title('Edad vs Conversión')
plt.show()

En este gráfico podemos ver que la mayoría de clientes (tanto los que convierten como los que no) están entre 30 y 50 años. La distribución de los que convierten (1) sigue una forma similar a los que no (0), pero parece haber ligeramente más peso en edades medias (30–45). En edades muy jóvenes (<25) y muy altas (>60) hay muy pocos casos de conversión.

In [ ]:
#Para analizar más a fondo la distribución de edades entre los clientes que se suscribieron, podemos filtrar a los clientes que tienen "Conversion" igual a 1, y luego graficar la distribución de edades.
df_conv = df_limpio[df_limpio['Conversion'] == 1]

sns.histplot(data=df_conv, x='age', bins=30, kde=True)
plt.title('Edad de clientes que SÍ convirtieron')
plt.show()

En este gráfico podemos deducir que la tasa de conversión es mayor entre las edades de 28 a 42 años.

In [ ]:
#En este gráfico comparamos la conversión por profesión, para ver si hay alguna profesión que tenga una mayor tasa de conversión que otras.

plt.figure(figsize=(10,5))
sns.countplot(data=df_limpio, x='job', hue='Conversion')
plt.xticks(rotation=45)
plt.title('Conversión por profesión')
plt.show()

En el gráfico podemos ver que la conversión es muy alta en personas con ocupación de administrador, tecnico y trabajadores manuales. Sin embargo, también representan las categorias donde mayor tasa de participación hubo. Es necesario, para poder hacer un análisis más preciso, medir la tasa de converción para evaluar la eficacia de la capaña.

In [ ]:
#Para poder realizar un gráfico más detallado de la conversión por profesión, podemos calcular la tasa de conversión para cada profesión, y luego graficar esa tasa de conversión.resumen_job
resumen_job = df_limpio.groupby('job').agg({
    'Conversion': ['mean', 'count']
})

resumen_job.columns = ['tasa_conversion', 'cantidad_clientes']
resumen_job = resumen_job.sort_values(by='tasa_conversion', ascending=False)

resumen_job

In [ ]:
#Luego de calcular la tasa de conversión por profesión, podemos graficar esa tasa de conversión junto con la cantidad de clientes para cada profesión.

fig, ax1 = plt.subplots(figsize=(12,6))

resumen_job['tasa_conversion'].plot(kind='bar', ax=ax1)
ax1.set_ylabel('Tasa de conversión')
ax1.set_title('Tasa de conversión y volumen por profesión')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(resumen_job.index, resumen_job['cantidad_clientes'], marker='o')
ax2.set_ylabel('Cantidad de clientes')

plt.show()

El análisis muestra que los estudiantes y jubilados presentan las tasas de conversión más altas, aunque con menor volumen de clientes. Por otro lado, perfiles como administrativos y técnicos, aunque con tasas de conversión moderadas, representan los segmentos más relevantes en términos absolutos debido a su gran volumen. Finalmente, grupos como trabajadores manuales y servicios muestran baja eficiencia, ya que concentran muchos contactos pero generan pocas conversiones, lo que indica una posible necesidad de optimizar o reducir esfuerzos en estos segmentos.

In [ ]:
#En el siguiente gráfico comparamos la conversión con los ingresos de los clientes, para ver si hay una relación directa entre ambos.

sns.boxplot(data=df_limpio, x='Conversion', y='Income')
plt.title('Income vs Conversión')
plt.show()

No se observan diferencias significativas en la distribución de ingresos entre los clientes que suscriben el depósito y los que no. Esto sugiere que el nivel de ingresos no es un factor determinante en la conversión dentro de esta campaña.

In [ ]:
#En el siguiente gráfico vamos a comprar la conversión por mes, para detectar algún patrón relacionado con la fecha.

plt.figure(figsize=(10,5))
sns.countplot(data=df_limpio, x='month_x', hue='Conversion')
plt.xticks(rotation=45)
plt.title('Conversión por mes')
plt.show()

In [ ]:
#En el siguiente gráfico vamos a comprar la conversión por año, para detectar algún patrón relacionado con la fecha.

plt.figure(figsize=(10,5))
sns.countplot(data=df_limpio, x='year_x', hue='Conversion')
plt.xticks(rotation=45)
plt.title('Conversión por año')
plt.show()

In [ ]:
#En el siguiente gráfico vamos a comprar la conversión por día de la semana, para detectar algún patrón relacionado con la fecha.

plt.figure(figsize=(10,5))
sns.countplot(data=df_limpio, x='weekday', hue='Conversion')
plt.xticks(rotation=45)
plt.title('Conversión por día de la semana')
plt.show()

A partir de todos estos gráficos, podemos deducir que no hay un patrón establecido en la fecha de conversión de los clientes (ni año, mes o día de la semana), por lo que no se puede segmentar por fechas más eficaces para aplicar la campaña.

In [ ]:
#En este gráfico, vamos a comprar al conversión con el número total de hijo que tiene cada cliente.

df_limpio['kids_total'] = df_limpio['Kidhome'] + df_limpio['Teenhome']

sns.boxplot(data=df_limpio, x='Conversion', y='kids_total')
plt.title('Número de hijos vs Conversión')
plt.show()

En este gráfico, tampoco vemos una relación clara entre el número total de hijos y la conversión, ya que la distribución de "kids_total" es similar tanto para los clientes que se suscribieron (Conversion=1) como para los que no se suscribieron (Conversion=0). Esto sugiere que el número de hijos no es un factor determinante en la decisión de suscribirse al producto o servicio ofrecido por el banco.

In [ ]:
#Para analizar las correlaciones entre las variables numéricas y poder ver de una forma simple y visual que variables tiene relaciones más fuertes, podemos crear un mapa de calor de correlaciones utilizando la función heatmap de seaborn.

corr = df_limpio.select_dtypes(include='number').corr()

plt.figure(figsize=(20,16))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Mapa de correlaciones')
plt.show()

En este mapa de calor, podemos ver que la correlación más fuerte para la variable que nos interesa es con la duración de la llamada, entre más tiempo de duración más tasa de conversión. Por otro lado, también vemos una relación inversamente proporcional muy fuerte entre el pday y la conversión, cuantos menos días pasaron desde el último contacto, mayor es la probabilidad de éxito. Finalmente, vemos que no hay una fuerte correlación entre el año, mes o día en el que el cliente se suscribió, por lo que no va a ser un factor determinante en el análisis. En cuanto al euribor, vemos que tiene una correlación negativa, cuando el eurobor baja (menor tasa de interés en el mercado) la conversión aumenta.

In [ ]:
sns.violinplot(data=df_limpio, x='Conversion', y='duration')
plt.title('Distribución de duración por conversión')
plt.show()

En este gráfico podemos validar lo indicado por el mapa de calor, ya que visualizamos como la duración de las llamadas de los clientes que se han suscripto tienen tendencia a ser más largas que aquellos que no se han suscripto.

In [ ]:
#Luego de identificar que la duración de la llamada tiene una fuerte correlación con la converssión, podemos segmentar la variable "duración" en categorias para analizar mejor su relación con la conversión. Luego, realizamos un gráfico de barras apiladas para visualizar la tasa de éxito por cada segmento de duración.

bins = [0, 60, 300, 900, df_limpio['duration'].max()]
labels = ['<1m', '1-5m', '5-15m', '>15m']
df_limpio['segmento_duracion'] = pd.cut(df_limpio['duration'], bins=bins, labels=labels)

tabla = pd.crosstab(df_limpio['segmento_duracion'], df_limpio['Conversion'], normalize='index')
tabla.plot(kind='bar', stacked=True, figsize=(8,5), title='Tasa de Éxito por Duración')

En este gráfico, podemos ver como la tasa de conversión en llamadas de menos de un minuto es nula y como aumenta a medida que la llamada es más larga. Además, pese a haber pocas llamadas de más de 15 minutos, la tasa de conversión es alta, por lo que, una llamada más larga no asegura la suscripción pero tiene una alta probabilidad de generar un nuevo cliente frente a llamadas de menos de 5 minutos.

In [ ]:
# Otro gráfico que nos sirve para ver la distribución de clientes por duración de llamada y su relación con la conversión, es un boxplot que compara la duración de las llamadas entre los clientes que se suscribieron (Conversion=1) y los que no se suscribieron (Conversion=0).

sns.boxplot(data=df_limpio, x='Conversion', y='duration')
plt.title('Duración de llamada vs Conversión')
plt.show()

En este gráfico vemos como la media de duración de llamadas de los clientes que convirtieron (1) es mayor que la de los clientes que no lo hicieron (0). Además, se pueden identificar de forma más clara los outliers o valores atípicos, pero aun así la tendencia de éxito se mantiene alta.

In [ ]:
#Para poder evaluar como el contexto económico impacta en la conversión, podemos analizar la variable "euribor3m", que es una tasa de interés de referencia en el mercado financiero. Podemos comparar la distribución de "euribor3m" entre los clientes que se suscribieron (Conversion=1) y los que no se suscribieron (Conversion=0) utilizando un boxplot.

sns.boxplot(data=df_limpio, x='Conversion', y='euribor3m')
plt.title('Euribor vs Conversión')
plt.show()

La información que podemos extraer tanto del mapa de calor como del boxplot en relación a la correlación entre el euribor y la tasa de conversión es muy útil, ya que podemos ver como los clientes son mucho más propensos a convertir cuando las tasas de interés están bajas. Esto lo podemos deducir viendo las mediandas de ambos grupos, en el grupo de conversión 0 la mediana está elevada, cerca de los 5 puntos y en el grupo de conversión 1 la mediana está cerca de 1.

In [ ]:
sns.swarmplot(data=df_limpio, x='Conversion', y='age')
plt.show()